# SV model with normal-mixture approximation

本 Notebook 演示如何在本地数据文件夹中加载分钟级期货数据，使用混合正态近似的随机波动（SV）模型进行快速 MCMC 演示。所有注释使用中文，print 输出和图表元素使用英文，便于跨平台显示。

In [ ]:
# 导入依赖，注释使用中文import numpy as npimport pandas as pdfrom pathlib import Pathfrom datetime import datetime# 可视化与进度显示import matplotlib.pyplot as pltimport seaborn as sns# 自定义模块from sv_toolkit.data import list_csv_files, load_contracts_in_dir, get_contract_symbol_from_pathfrom sv_toolkit.mcmc import run_mcmc_svfrom sv_toolkit.batch import make_timestamped_root, save_param_summaryfrom sv_toolkit.plotting import (    plot_returns,    plot_volatility,    plot_return_histogram,    plot_acf_returns,    plot_intraday_pattern,    plot_param_posterior,    plot_vol_and_abs_returns,    plot_standardized_residuals,    plot_mixture_usage,)# 设置 matplotlib 显示风格sns.set_style('whitegrid')plt.rcParams['figure.dpi'] = 120# 确定数据目录，默认为当前仓库下的 2005年__20250905 文件夹data_dir = Path('../2005年__20250905')# 创建带时间戳的输出目录，方便保存图片与参数output_root = make_timestamped_root(Path('outputs'))fig_dir = output_root / 'demo'fig_dir.mkdir(parents=True, exist_ok=True)print(f'Environment ready. Data dir: {data_dir}. Output root: {output_root}')

## 1. 浏览数据文件



In [ ]:
# 列出数据目录中的前 5 个文件，避免一次性打印全部
preview_files = list_csv_files(data_dir, max_files=5)
print('Preview finished.')

## 2. 读取单个/少量文件并构造收益率



In [ ]:
# 选择数据子集，按合约独立加载，便于快速跑通流程datasets = load_contracts_in_dir(    data_dir,    contract_code=None,  # 可以填入具体合约代码，比如 'A0505.XDCE'    start_time=None,     # 可以填入 '2005-01-01' 这样的字符串    end_time=None,       # 可以填入 '2008-12-31' 等字符串    max_files=1,         # 只读取一个文件进行示范    max_rows_per_file=10000,  # 限制行数，方便快速运行)symbols = list(datasets.keys())if not symbols:    raise RuntimeError('No contracts loaded from the provided directory.')contract_tag = symbols[0]contract_data = datasets[contract_tag]r = contract_data['r']y_star = contract_data['y_star']df = contract_data['df']print(f'Selected contract {contract_tag} with returns length = {len(r)} and dataframe rows = {len(df)}')

## 3. 运行简化版 MCMC



In [ ]:
# 运行一个轻量级的 MCMC 样例，确保进度信息可见
mcmc_results = run_mcmc_sv(
    r=r,
    y_star=y_star,
    n_iter=120,      # 演示用较小迭代次数
    burn_in=40,
    thin=2,
    rng_seed=2025,
    progress_every=20,
)

print('Sampling finished. Posterior samples available in mcmc_results dict.')

## 4. 保存参数摘要到时间戳目录



In [ ]:
# 将后验参数统计保存为 CSV，同时记录运行信息，所有输出带时间戳extra_info = {    'T': len(r),    'n_iter': 120,    'burn_in': 40,    'thin': 2,    'data_dir': str(data_dir),    'contract_tag': contract_tag,}save_param_summary(mcmc_results, fig_dir, contract_tag, extra_info=extra_info)print(f'Parameter summary saved under: {fig_dir}')

## 5. 绘制收益率、隐含波动率与其他诊断图



In [ ]:
# 绘制收益率轨迹并保存returns_png = plot_returns(df, fig_dir, title_suffix=contract_tag)print(f'Returns figure saved to: {returns_png}')# 绘制波动率轨迹：使用后验样本 h 计算if len(mcmc_results['h']) > 0:    vol_png = plot_volatility(mcmc_results['h'], df, fig_dir, title_suffix=contract_tag)    print(f'Volatility figure saved to: {vol_png}')else:    print('No h samples available to plot volatility.')# 更多诊断图：直方图、ACF、日内模式、参数后验、波动率对比、标准化残差、混合频率hist_png = plot_return_histogram(r, fig_dir, title_suffix=contract_tag)acf_png = plot_acf_returns(r, fig_dir, title_suffix=contract_tag)intra_png = plot_intraday_pattern(df, fig_dir, title_suffix=contract_tag)param_png = plot_param_posterior(mcmc_results, fig_dir, title_suffix=contract_tag)vol_abs_png = plot_vol_and_abs_returns(mcmc_results['h'], df, fig_dir, title_suffix=contract_tag)std_resid_png = plot_standardized_residuals(r, mcmc_results, fig_dir, title_suffix=contract_tag)mix_png = Noneif 's' in mcmc_results and len(mcmc_results['s']) > 0:    mix_png = plot_mixture_usage(mcmc_results['s'], fig_dir, title_suffix=contract_tag)print('Additional figures:', hist_png, acf_png, intra_png, param_png, vol_abs_png, std_resid_png, mix_png)